In [1]:
import numpy as np
import torch

from spherical import Wigner

from pytorch3d.transforms import random_rotations, random_quaternions, matrix_to_euler_angles, matrix_to_quaternion

In [3]:
torch.load('/pscratch/sd/z/zhantao/neurorient_repo/data/PR772/PR772_neurorient_downsampled_128x128.pt')['intensities'].max()

tensor(1433.)

In [4]:
torch.load('/pscratch/sd/z/zhantao/neurorient_repo/data/PR772/PR772_neurorient_downsampled_128x128_filtered.pt')['intensities'].max()

tensor(601.0857)

In [11]:
torch.log(torch.ones(1) * 1000).clamp(9, 16)

tensor([9.])

In [5]:
torch.log(torch.exp(torch.ones(1)))

tensor([1.])

In [170]:
class WignerD6Basis:
    def __init__(self,):
        self.wigner = Wigner(6,6)
        
    def m2i(self, m):
        return m + 6
    
    def i2m(self, i):
        return i - 6
        
    def evaluate_spha_coefficients(self, q):
        device = q.device
        dtype = q.dtype
        q = q.detach().cpu().numpy()
        D_mat = self.wigner.D(q).reshape(-1, 13, 13)
        D_out_complex = 1/5 * (
            - np.sqrt(7)  * D_mat[:,self.m2i( 5),:]
            + np.sqrt(11) * D_mat[:,self.m2i( 0),:]
            + np.sqrt(7)  * D_mat[:,self.m2i(-5),:]
        )
        D_out_complex = torch.from_numpy(D_out_complex).to(device)
        D_out = torch.zeros(q.shape[0], 13, device=device, dtype=dtype)
        D_out[:,:6] = np.sqrt(2) * D_out_complex.imag[:,:6]
        D_out[:,6]  = D_out_complex.real[:,6]
        D_out[:,7:] = np.sqrt(2) * D_out_complex.real[:,7:]
        return D_out
        
    # def evaluate_basis_func_values(self, theta, phi):
        

In [216]:
# rotmat_input = random_rotations(1)
rotmat_input = torch.eye(3).unsqueeze(0)
quat_input = matrix_to_quaternion(rotmat_input)

In [217]:
wigner_basis = WignerD6Basis()

In [218]:
wigner_basis.evaluate_spha_coefficients(quat_input)

tensor([[ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.6633,  0.0000,
          0.0000,  0.0000,  0.0000, -0.7483,  0.0000]])

In [219]:
xx, yy, zz = torch.meshgrid(torch.linspace(-1,1,101), torch.linspace(-1,1,101), torch.linspace(-1,1,101), indexing='ij')

In [220]:
xyz_grid = torch.cat([xx[...,None], yy[...,None], zz[...,None]], dim=-1)

xyz_grid = xyz_grid / xyz_grid.norm(dim=-1, keepdim=True)

In [221]:
from neurorient.external.rsh import rsh_cart_6, rsh_cart_5

In [222]:
Ylm = rsh_cart_6(xyz_grid)[...,-13:]

In [223]:
volume = torch.einsum('...i, i -> ...', Ylm, wigner_basis.evaluate_spha_coefficients(quat_input)[0])

In [224]:
import matplotlib.pyplot as plt
